# 订单计算本体

## 1 概念

| 概念     | 含义                                           |
| -------- | ---------------------------------------------- |
| 订单     | 客户销售订单或询单，包含订单头信息             |
| 订单明细 | 订单的具体行项，记录需要交付的货品、数量和交期 |
| 货品     | 企业生产输出的物料，包含成品和半成品、         |
| BOM      | 生产该货品所需的物料清单                       |
| 排产任务 | 对订单进行生产排程的任务，定义排产策略和目标   |
| 排查结果 | 排产任务执行后产出的调度方案及关键指标     |
| 欠料明细 | 下发排产结果与当前库存对比后得出的缺料清单     |

## 2 关系

| 源概念   | 关系名               | 目标概念 | 基数 | 说明                                                      |
| -------- | -------------------- | -------- | ---- | --------------------------------------------------------- |
| 订单     | hasDetail            | 订单明细 | 1:N  | 1个订单含多个明细项                                       |
| 订单明细 | refersToProduct      | 货品     | N:1  | 多个明细项可指向同一货品                                  |
| 货品     | hasBOM               | BOM      | 1:1  | 1个货品有1个BOM                                           |
| 订单     | drivesScheduling     | 排产任务 | 1:N  | 1个订单可以产生多个排产任务                               |
| 排产任务 | hasVersionDerivative | 排产任务 | 1:N  | 1个排产任务可衍生多个版本或场景排产任务，表示版本派生关系 |
| 排产任务 | producesPlan         | 排产结果 | 1:1  | 1个排产任务产出1个排产结果                                |
| 排产结果 | hasShortageDetail    | 欠料明细 | 1:N  | 1个排产结果对应多个原料欠料明细                           |

## 3 属性

### 3.1 订单

| 字段         | 类型   | 约束        |
| ------------ | ------ | ----------- |
| 订单编号     | 字符串 |             |
| 订单名称     | 字符串 |             |
| 订单批次号   | 字符串 |             |
| 订单缓冲周期 | 整数   | ≥ 0         |
| 订单类型     | 枚举   | 销售 / 询单 |

### 3.2 订单明细

| 字段     | 类型   | 约束                             |
| -------- | ------ | -------------------------------- |
| 订单编号 | 字符串 |                                  |
| 货品编号 | 字符串 |                                  |
| 货品名称 | 字符串 |                                  |
| 起排日期 | 日期   | yyyy-mm-dd 格式                  |
| 截止日期 | 日期   | yyyy-mm-dd 格式，> 起排日期      |
| 交付数量 | 整数   | > 0                              |
| 优先级   | 枚举   | 按期交付 / 紧急插单 / 可延期交付 |

### 3.3 货品

| 字段           | 类型   | 约束                   |
| -------------- | ------ | ---------------------- |
| 货品编号       | 字符串 |                        |
| 货品名称       | 字符串 |                        |
| 货品大类       | 枚举   | 成品 / 半成品          |
| 物料类型       | 枚举   | 主料 / 可回收料 / 废料 |
| 主单位         | 字符串 |                        |
| 副单位         | 字符串 |                        |
| 主副单位换算比 | 浮点数 | > 0                    |
| 管理部门       | 字符串 |                        |
| 生产部门       | 字符串 |                        |
| 预设仓库       | 字符串 |                        |
| 规格           | 字符串 |                        |

### 3.4 BOM

| 字段     | 类型   | 约束 |
| -------- | ------ | ---- |
| BOM编号  | 字符串 |      |
| BOM名称  | 字符串 |      |
| 货品编号 | 字符串 |      |
| 货品名称 | 字符串 |      |
| BOM版本  | 字符串 |      |

### 3.5 排产任务

| 字段           | 类型   | 约束                                         |
| -------------- | ------ | -------------------------------------------- |
| 订单编号       | 字符串 |                                              |
| 订单名称       | 字符串 |                                              |
| 排产任务编号   | 字符串 |                                              |
| 排产任务名称   | 字符串 |                                              |
| 排产场景       | 字符串 |                                              |
| 排产方式       | 枚举   | 正排 / 倒排                                  |
| 设备利用率目标 | 枚举   | 设备利用率 / 设备组均衡程度                  |
| 订单交期目标   | 枚举   | 订单延期率最低 / 订单平均延期最小            |
| 加工成本最低   | 枚举   | 启用 / 关闭                                  |
| 使用剩余产能   | 枚举   | 启用 / 关闭                                  |
| 附件转换率最低 | 枚举   | 启用 / 关闭                                  |
| 任务状态       | 枚举   | 已下发 / 已排产 / 数据异常 / 等待提交 / 其他 |

### 3.6 排产结果

| 字段             | 类型   | 约束            |
| ---------------- | ------ | --------------- |
| 排产任务编号     | 字符串 |                 |
| 排产任务名称     | 字符串 |                 |
| 排产场景         | 字符串 |                 |
| 方案名称     | 字符串 |                 |
| 方案状态     | 字符串 | 已下发 / 未锁定 |
| 总产品数量       | 整数   | > 0             |
| 总工序数量       | 整数   | > 0             |
| 平均生产周期     | 浮点数 | > 0             |
| 延期工件数量     | 整数   | ≥ 0             |
| 延期率           | 浮点数 | 0-100           |
| 总加工成本       | 浮点数 | ≥ 0             |
| 总利润           | 浮点数 |                 |
| 换型总次数       | 整数   | ≥ 0             |
| 设备平均利用率   | 浮点数 | 0-1             |
| 设备平均工作时间 | 浮点数 | ≥ 0（小时）     |
| 人员平均工作时间 | 浮点数 | ≥ 0（小时）     |
| 最早开始时间     | 日期   | yyyy-mm-dd 格式 |
| 最晚结束时间     | 日期   | yyyy-mm-dd 格式 |

### 3.7 欠料明细

| 字段         | 类型       | 约束              |
| ------------ | ---------- | ----------------- |
| 排产任务编号 | 字符串     |                   |
| 排产任务名称 | 字符串     |                   |
| 排产场景     | 字符串     |                   |
| 原材料编号   | 字符串     |                   |
| 原材料名称   | 字符串     |                   |
| 欠料结果     | list[dict] | [{'日期':'数量'}] |

## 4 行为

### 4.1 创建订单

| 项目     | 内容                                                                                                                 |
| -------- | -------------------------------------------------------------------------------------------------------------------- |
| 行为时间 | yyyy-mm-dd hh:mm:ss                                                                                                  |
| 行为类型 | CREATE                                                                                                               |
| 行为接口 | /api/orders                                                                                                          |
| 接口参数 | 订单名称, 订单批次号, 订单类型, 订单明细                                                                             |
| 行为描述 | 用户或智能体创建新的订单及其明细                                                                                     |
| 前置约束 | 订单名称, 订单批次号, 订单类型, 订单明细不为空；订单明细里，货品编号存在且具有BOM，付数量不为0，截止日期大于起排日期 |
| 后置约束 |                                                                                                                      |
| 返回结果 | 创建的订单及明细信息                                                                                                 |
| 产生事件 | 订单创建事件                                                                                                         |

### 4.2 订单更新

| 项目     | 内容                                           |
| -------- | ---------------------------------------------- |
| 行为时间 | yyyy-mm-dd hh:mm:ss                            |
| 行为类型 | UPDATE                                         |
| 行为接口 | /api/orders                                    |
| 接口参数 | 订单编号/订单名称/订单批次号/订单类型/订单明细 |
| 行为描述 | 用户或智能体更新已有订单的头信息或明细项       |
| 前置约束 | 订单编号不为空，订单已创建但未创建排产任务     |
| 后置约束 |                                                |
| 返回结果 | 更新的订单及明细信息                           |
| 产生事件 | 订单更新事件                                   |

### 4.3 订单删除

| 项目     | 内容                                       |
| -------- | ------------------------------------------ |
| 行为时间 | yyyy-mm-dd hh:mm:ss                        |
| 行为类型 | DELETE                                     |
| 行为接口 | /api/orders                                |
| 接口参数 | 订单编号                                   |
| 行为描述 | 用户或智能体删除已有订单及明细             |
| 前置约束 | 订单编号不为空，订单已创建但未创建排产任务 |
| 后置约束 |                                            |
| 返回结果 | 删除的订单及明细信息                       |
| 产生事件 | 删除订单事件                               |

### 4.4 查询订单

| 项目     | 内容                           |
| -------- | ------------------------------ |
| 行为时间 | yyyy-mm-dd hh:mm:ss            |
| 行为类型 | READ                           |
| 行为接口 | /api/orders                    |
| 接口参数 | 订单编号，订单名称             |
| 行为描述 | 用户或智能体查询订单及明细信息 |
| 前置约束 | 订单编号 或 订单名称不为空     |
| 后置约束 |                                |
| 返回结果 | 订单编号等明细信息             |
| 产生事件 | 无                             |

### 4.5 创建排产任务

| 项目     | 内容                                                     |
| -------- | -------------------------------------------------------- |
| 行为时间 | yyyy-mm-dd hh:mm:ss                                      |
| 行为类型 | CREATE                                                   |
| 行为接口 | /api/scheduling/tasks                                    |
| 接口参数 | 订单编号，算法参数                                       |
| 行为描述 | 用户或智能体根据订单信息创建排产任务，并设定排产算法参数 |
| 前置约束 | 订单编号不为空                                           |
| 后置约束 |                                                          |
| 返回结果 | 排产任务编号                                             |
| 产生事件 | 排产任务创建事件                                         |

### 4.6 新增排产场景

| 项目     | 内容                                                                                   |
| -------- | -------------------------------------------------------------------------------------- |
| 行为时间 | yyyy-mm-dd hh:mm:ss                                                                    |
| 行为类型 | CREATE                                                                                 |
| 行为接口 | /api/scheduling/scenarios                                                              |
| 接口参数 | 排产任务编号, 算法参数                                                                 |
| 行为描述 | 基于已有排产任务创建新的排产场景，传入需要修改的相关算法参数，返回新场景的排产任务编号 |
| 前置约束 | 排产任务编号scheduleId不为空，且其对应的排产任务状态显示已排产                         |
| 后置约束 |                                                                                        |
| 返回结果 | 新的排产任务编号scheduleId                                                             |
| 产生事件 | 新增排产场景事件                                                                       |

### 4.7 启动排产任务

| 项目     | 内容                           |
| -------- | ------------------------------ |
| 行为时间 | yyyy-mm-dd hh:mm:ss            |
| 行为类型 | EXECUTE                        |
| 行为接口 | /api/scheduling/execute        |
| 接口参数 | 排产任务编号scheduleId         |
| 行为描述 | 用户或智能体选择排产任务并执行 |
| 前置约束 | 排产任务编号scheduleId不为空   |
| 后置约束 |                                |
| 返回结果 | 返回启动成功或失败信息         |
| 产生事件 | 启动排产任务事件               |

### 4.8 查询排产任务状态

| 项目     | 内容                                                                   |
| -------- | ---------------------------------------------------------------------- |
| 行为时间 | yyyy-mm-dd hh:mm:ss                                                    |
| 行为类型 | READ                                                                   |
| 行为接口 | /api/scheduling/tasks/status                                           |
| 接口参数 | 排产任务编号scheduleId，订单编号                                       |
| 行为描述 | 用户或智能体查询排产任务的当前状态                                     |
| 前置约束 | 排产任务编号scheduleId或订单编号不为空                                 |
| 后置约束 |                                                                        |
| 返回结果 | 排产任务状态信息，包括任务状态（已下发/已排产/数据异常/等待提交/其他） |
| 产生事件 | 无                                                                     |

### 4.9 查询排产任务结果

| 项目     | 内容                                                                     |
| -------- | ------------------------------------------------------------------------ |
| 行为时间 | yyyy-mm-dd hh:mm:ss                                                      |
| 行为类型 | READ                                                                     |
| 行为接口 | /api/scheduling/results                                                  |
| 接口参数 | 排产任务编号scheduleId，订单编号                                         |
| 行为描述 | 用户或智能体查询排产任务结果                                             |
| 前置约束 | 排产任务编号scheduleId或订单编号不为空                                   |
| 后置约束 |                                                                          |
| 返回结果 | 该排产任务的结果列表，如排产任务编号，排产任务名称，延期率，设备利用率等 |
| 产生事件 | 无                                                                       |

### 4.9 启动排产任务查询定时任务

| 项目     | 内容                                                                                                               |
| -------- | ------------------------------------------------------------------------------------------------------------------ |
| 行为时间 | yyyy-mm-dd hh:mm:ss                                                                                                |
| 行为类型 | CREATE                                                                                                             |
| 行为接口 | /api/scheduling/poll/start                                                                                         |
| 接口参数 | 排产任务编号scheduleId, 轮询间隔intervalMinutes, 检查次数checkCnt                                                  |
| 行为描述 | 后台启动定时任务，按指定间隔和scheduleId轮询排产任务状态和执行结果，当计算结果查询到或者轮训次数超过阈值后自动结束 |
| 前置约束 | 排产任务已创建并提交执行，返回了scheduleId                                                                         |
| 后置约束 |                                                                                                                    |
| 返回结果 | 轮询获取的排产状态和结果，包括scheduleTaskNumber/scheduleTaskName/schedulingScenario等信息；或者结果获取失败信息； |
| 产生事件 | 排产任务查询启动；排产任务查询执行完毕；                                                                           |

### 4.9 排产结果下发

| 项目     | 内容                                                                               |
| -------- | ---------------------------------------------------------------------------------- |
| 行为时间 | yyyy-mm-dd hh:mm:ss                                                                |
| 行为类型 | UPDATE                                                                             |
| 行为接口 | /api/scheduling/tasks/deploy                                                       |
| 接口参数 | 排产任务编号scheduleId，订单编号                                                   |
| 行为描述 | 用户或智能体将已成功执行的排产任务下发                                             |
| 前置约束 | 排产任务编号scheduleId或订单编号不为空，排产结果已存在且方案状态为"未锁定" |
| 后置约束 |                                                                                    |
| 返回结果 | 下发成功或下发失败信息                                                             |
| 产生事件 | 排产结果下发事件                                                               |

### 4.10 排产结果撤回

| 项目     | 内容                                                                               |
| -------- | ---------------------------------------------------------------------------------- |
| 行为时间 | yyyy-mm-dd hh:mm:ss                                                                |
| 行为类型 | UPDATE                                                                             |
| 行为接口 | /api/scheduling/plans/deploy/cancel                                                |
| 接口参数 | 排产任务编号scheduleId，订单编号                                                   |
| 行为描述 | 用户或智能体取消已下发的排产结果，方案状态回退为"未锁定"                       |
| 前置约束 | 排产任务编号scheduleId或订单编号不为空，方案结果已存在且方案状态为"已下发" |
| 后置约束 |                                                                                    |
| 返回结果 | 排产结果取消成功或失败信息                                                         |
| 产生事件 | 排产结果撤回事件                                                               |

### 4.11 查询排产结果欠料明细

| 项目     | 内容                                                            |
| -------- | --------------------------------------------------------------- |
| 行为时间 | yyyy-mm-dd hh:mm:ss                                             |
| 行为类型 | READ                                                            |
| 行为接口 | /api/material/shortages                                         |
| 接口参数 | 排产任务编号scheduleId，订单编号                                |
| 行为描述 | 用户或智能体查询订单下发后的齐套欠料头信息                      |
| 前置约束 | 排产任务编号scheduleId或订单编号不为空；排产结果显示已下发  |
| 后置约束 |                                                                 |
| 返回结果 | 下发排产结果信息，包括订单名称/排产任务信息等，以及齐套欠料列表 |
| 产生事件 | 无                                                              |

## 5 规则

### 5.1 验证规则

| 编号 | 规则名                | 规则描述                              | 涉及概念       |
| ---- | --------------------- | ------------------------------------- | -------------- |
| V01  | 订单明细-订单存在性   | 订单明细.订单编号 必须存在于 订单     | 订单明细、订单 |
| V02  | 订单明细-货品存在性   | 订单明细.货品编号 必须存在于 货品     | 订单明细、货品 |
| V03  | 订单明细-截止日期约束 | 订单明细.截止日期 > 订单明细.起排日期 | 订单明细       |
| V04  | 订单明细-交付数量约束 | 订单明细.交付数量 > 0                 | 订单明细       |

### 5.2 推理规则

| 编号 | 规则名                           | 规则描述                                                                                                | 涉及概念               |
| ---- | -------------------------------- | ------------------------------------------------------------------------------------------------------- | ---------------------- |
| I01  | 已排产完成后可创建新场景排产任务 | 排产任务.任务状态 = "已排产" → 可以此排产任务为基础新增场景任务，可调整算法参数后重新计算               | 排产任务、排产结果     |
| I02  | 排产结果成功下发后提示查欠料     | 排产结果.方案状态 = "已下发" → 提示用户或智能体查询该排产结果的欠料明细                             | 排产结果、欠料明细     |
| I03  | 欠料查出后提示创建采购单         | 欠料明细查询到，且该订单未关联采购单或关联采购单不满足欠料需求 → 提示用户或智能体是否创建原材料采购单   | 订单、欠料明细、采购单 |
| I04  | 订单创建后提示排产               | 订单创建成功 → 提示用户或智能体可对该订单创建排产任务                                                   | 订单、排产任务         |
| I05  | 排产任务创建后提示启动           | 排产任务创建成功 → 提示用户或智能体可启动该排产任务                                                     | 排产任务               |
| I06  | 排产启动后提示等待               | 排产任务启动 → 提示用户或智能体需等待结果，已自动启动定时任务轮询；也可主动发起查询排产结果和排产状态； | 排产任务、排产结果     |
| I07  | 排产结果撤回后提示撤采购         | 排产结果撤回 → 若已有采购记录，提示用户或智能体是否撤回相关采购申请                                     | 排产结果、采购单       |

## 6 事件

### 6.1 订单创建事件

| 项目     | 内容                                                |
| -------- | --------------------------------------------------- |
| 事件编号 | E001                                                |
| 事件名称 | 订单创建事件                                        |
| payload  | 订单编号，订单名称，订单类型，订单明细；status；msg |
| 后置动作 | 无                                                  |

### 6.2 订单更新事件

| 项目     | 内容                                                |
| -------- | --------------------------------------------------- |
| 事件编号 | E002                                                |
| 事件名称 | 订单更新事件                                        |
| payload  | 订单编号，订单名称，订单类型，订单明细；status；msg |
| 后置动作 | 无                                                  |

### 6.3 订单删除事件

| 项目     | 内容                                                |
| -------- | --------------------------------------------------- |
| 事件编号 | E003                                                |
| 事件名称 | 订单删除事件                                        |
| payload  | 订单编号，订单名称，订单类型，订单明细；status；msg |
| 后置动作 | 无                                                  |

### 6.4 排产任务创建事件

| 项目     | 内容                                          |
| -------- | --------------------------------------------- |
| 事件编号 | E004                                          |
| 事件名称 | 排产任务创建事件                              |
| payload  | 排产任务编号, 订单编号, 排产方式；status；msg |
| 后置动作 | 无                                            |

### 6.5 新增排产场景事件

| 项目     | 内容                            |
| -------- | ------------------------------- |
| 事件编号 | E005                            |
| 事件名称 | 新增排产场景事件                |
| payload  | 订单编号，算法参数；status；msg |
| 后置动作 | 无                              |

### 6.6 排产任务启动事件

| 项目     | 内容                                   |
| -------- | -------------------------------------- |
| 事件编号 | E0056                                  |
| 事件名称 | 排产任务启动事件                       |
| payload  | 排产任务编号, 排产方式；status；msg    |
| 后置动作 | 若成功，则主动启动排产任务查询定时任务 |

### 6.7 排产任务定时查询执行完成事件

| 项目     | 内容                                                                        |
| -------- | --------------------------------------------------------------------------- |
| 事件编号 | E006                                                                        |
| 事件名称 | 排产执行完成事件                                                            |
| payload  | 排产任务编号，排产任务名称，方案名称，延期率，设备利用率等；status；msg |
| 后置动作 | 无                                                                          |

### 6.8 排产任务定时查询启动事件

| 项目     | 内容                                          |
| -------- | --------------------------------------------- |
| 事件编号 | E007                                          |
| 事件名称 | 排产任务定时查询启动事件                      |
| payload  | 排产任务编号, 轮询间隔, 轮询次数；status；msg |
| 后置动作 | 无                                            |

### 6.9 排产结果下发事件

| 项目     | 内容                                                        |
| -------- | ----------------------------------------------------------- |
| 事件编号 | E008                                                        |
| 事件名称 | 排产结果下发事件                                            |
| payload  | 排产任务编号, 排产任务名称, 方案名称, 方案状态；status；msg |
| 后置动作 | 若成功，查询排产结果欠料明细，并查询该订单的欠料采购情况    |

### 6.10 排产结果撤回事件

| 项目     | 内容                                                        |
| -------- | ----------------------------------------------------------- |
| 事件编号 | E009                                                        |
| 事件名称 | 排产结果撤回事件                                            |
| payload  | 排产任务编号, 排产任务名称, 方案名称, 方案状态；status；msg |
| 后置动作 | 若成功，查询该订单的欠料采购情况                            |
